# Análisis de Resultados — Benchmark de Detección YOLO
## Proyecto: Estimación de velocidad vehicular en sistema embebido — TFG CE-TEC
Análisis exploratorio de los resultados del benchmark para seleccionar la variante YOLO
más adecuada para despliegue en hardware embebido (Raspberry Pi 4 / Jetson Nano).

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Carga automática de los resultados más recientes en results/benchmarks/
RESULTS_DIR = Path("../results/benchmarks")

csv_files = sorted(RESULTS_DIR.glob("frame_data_*.csv"))
json_files = sorted(RESULTS_DIR.glob("summary_*.json"))

if not csv_files or not json_files:
    raise FileNotFoundError(
        "No se encontraron resultados en results/benchmarks/. "
        "Ejecuta primero: python scripts/run_benchmark.py"
    )

latest_csv = csv_files[-1]
latest_json = json_files[-1]

print(f"CSV más reciente : {latest_csv.name}")
print(f"JSON más reciente: {latest_json.name}")

df = pd.read_csv(latest_csv)

with open(latest_json, "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\nPrimeras 5 filas del detalle por frame:")
display(df.head())

print("\nResumen del benchmark (system_info):")
summary["system_info"]

In [ ]:
# Estadísticas descriptivas por modelo
columns_of_interest = ["fps", "inference_ms", "vehicle_count", "cpu_percent", "ram_mb"]
stats = df.groupby("model_id")[columns_of_interest].describe()
stats

In [ ]:
# Distribución de FPS por modelo
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="model_id", y="fps", hue="model_id", palette="viridis", legend=False)
plt.title("Distribución de FPS por modelo")
plt.xlabel("Modelo")
plt.ylabel("FPS")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vehículos detectados por clase y modelo
class_cols = ["car_count", "truck_count", "bus_count", "motorcycle_count"]
class_means = df.groupby("model_id")[class_cols].mean()
class_means.columns = ["Auto", "Camión", "Bus", "Moto"]

class_means.plot(kind="bar", figsize=(10, 6), color=["#2ecc71", "#3498db", "#e67e22", "#f1c40f"])
plt.title("Promedio de vehículos detectados por clase y modelo")
plt.xlabel("Modelo")
plt.ylabel("Vehículos detectados (promedio por frame)")
plt.legend(title="Clase")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa para el Capítulo 4 del informe
models_df = pd.DataFrame(summary["models"])

tabla_informe = models_df[[
    "model_id", "parameters_M", "fps_mean", "fps_p5",
    "inference_ms_mean", "ram_mb_mean", "cpu_percent_mean",
    "vehicles_per_frame_mean",
]].copy()

tabla_informe.columns = [
    "Modelo", "Parámetros (M)", "FPS medio", "FPS p5",
    "Inf. (ms)", "RAM (MB)", "CPU (%)", "Veh./frame",
]
tabla_informe = tabla_informe.sort_values("FPS medio", ascending=False).reset_index(drop=True)

output_path = Path("../results/benchmarks/tabla_informe.csv")
tabla_informe.to_csv(output_path, index=False)
print(f"Tabla exportada a: {output_path}")

display(tabla_informe.style.highlight_max(axis=0))

## Conclusiones del benchmark

**Modelo seleccionado:** [completar]

**Justificación:**
- El modelo [X] ofrece el mejor balance entre FPS y precisión para las restricciones de hardware embebido.
- [Completar con los datos observados]

**Limitaciones de este benchmark:**
- Las pruebas se realizaron sobre [CPU/GPU local], no sobre el hardware embebido objetivo.
- [Completar]